In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [8]:
B = 4
D = 3

image_features = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
])

text_features = torch.tensor([
    [0.9, 0.1, 0.0],
    [0.1, 0.9, 0.0],
    [0.0, 0.1, 0.9],
    [0.8, 0.8, 0.1],
])

image_features @ text_features.T should be 4,4, 
mat mul the 4 images with the 4 texts, each having 3 features.
Multiply each feature for pairwise number representing
the mat mul'd image and text. 2,1 should represent the matrix multiplied
version of the features of the 3rd image with the features of the 
2nd text example. It normalizes so that it can just filter for 
semantic directional alignment rather than magnitude. 

In [14]:
image_features_norm = torch.nn.functional.normalize(image_features)
text_features_norm = torch.nn.functional.normalize(text_features)

print(image_features_norm)
print(text_features_norm)

tensor([[1.0000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 1.0000],
        [0.7071, 0.7071, 0.0000]])
tensor([[0.9939, 0.1104, 0.0000],
        [0.1104, 0.9939, 0.0000],
        [0.0000, 0.1104, 0.9939],
        [0.7044, 0.7044, 0.0880]])


In [17]:
similarity_it = image_features_norm @ text_features_norm.T
similarity_ti = similarity_it.T

print(similarity_it)
print(similarity_ti)

tensor([[0.9939, 0.1104, 0.0000, 0.7044],
        [0.1104, 0.9939, 0.1104, 0.7044],
        [0.0000, 0.0000, 0.9939, 0.0880],
        [0.7809, 0.7809, 0.0781, 0.9961]])
tensor([[0.9939, 0.1104, 0.0000, 0.7809],
        [0.1104, 0.9939, 0.0000, 0.7809],
        [0.0000, 0.1104, 0.9939, 0.0781],
        [0.7044, 0.7044, 0.0880, 0.9961]])


Softmax softens the distribution too much, we need to add temperature to control sharpness.

In [19]:
targets = torch.arange(B)

F.cross_entropy(similarity_it, targets)
probs = similarity_it.softmax(dim=1)

print(similarity_it[0])
print(probs[0])

tensor([0.9939, 0.1104, 0.0000, 0.7044])
tensor([0.3949, 0.1632, 0.1462, 0.2957])


In [23]:
logit_scale = torch.tensor(2.6592)  # roughly log(1 / 0.07)

scaled_similarity = logit_scale.exp() * similarity_it

print("scale:", logit_scale.exp())
print("raw probs:", similarity_it[0].softmax(dim=0))
print("scaled probs:", scaled_similarity[0].softmax(dim=0))

scale: tensor(14.2849)
raw probs: tensor([0.3949, 0.1632, 0.1462, 0.2957])
scaled probs: tensor([9.8426e-01, 3.2533e-06, 6.7177e-07, 1.5738e-02])


In [31]:
def clip_loss(
    image_features: torch.Tensor,
    text_features: torch.Tensor,
    logit_scale: torch.Tensor,
) -> torch.Tensor:

    # 1. normalize image features
    image_features_norm = torch.nn.functional.normalize(image_features)
    
    # 2. normalize text features
    text_features_norm = torch.nn.functional.normalize(text_features)

    # 3. construct B x B similarity matrix
    similarity_it = image_features_norm @ text_features_norm.T
    similarity_ti = similarity_it.T

    # 4. construct targets
    batch_size = image_features.shape[0]
    targets = torch.arange(batch_size, device=image_features.device)

    # 5. image -> text cross entropy
    scale = logit_scale.exp()
    logits_it = scale * similarity_it
    ce_it = F.cross_entropy(logits_it, targets)
    
    # 6. text -> image cross entropy
    logits_ti = scale * similarity_ti
    ce_ti = F.cross_entropy(logits_ti, targets)

    # 7. symmetric average
    loss = (ce_it + ce_ti) / 2

    return loss

In [32]:
loss = clip_loss(
    image_features,
    text_features,
    torch.tensor(2.6592),
)

print(loss)

tensor(0.0305)


In [33]:
shuffled_text_features = text_features[[2, 0, 3, 1]]

shuffled_loss = clip_loss(
    image_features,
    shuffled_text_features,
    torch.tensor(2.6592),
)

print("aligned loss:", loss.item())
print("shuffled loss:", shuffled_loss.item())

aligned loss: 0.030478760600090027
shuffled loss: 10.738485336303711
